In [1]:
from pathlib import Path
import polars as pl

shared = Path("~/shared/birdclef").expanduser()
perch = pl.scan_parquet(
    f"{shared}/data/2025/train_audio-infer-soundscape/Perch/parts/embed/*.parquet"
)
columns = perch.collect_schema().names()
perch = (
    perch.select(
        "file",
        "start_time",
        "end_time",
        (
            pl.concat_list(columns[3:])
            .list.to_array(len(columns[3:]))
            .alias("embedding")
        ),
    )
    .sort("file", "start_time")
    .with_columns(
        species=pl.col("file").str.split("/").list.get(-2),
        track=pl.col("file").str.split("/").list.get(-1).str.split(".").list.first(),
    )
)
# count and schema
display(perch.collect_schema())

Schema([('file', String),
        ('start_time', Float64),
        ('end_time', Float64),
        ('embedding', Array(Float64, shape=(1280,))),
        ('species', String),
        ('track', String)])

In [4]:
list1 = [
    # Amphibia
    "65344",  # Boettger's Colombian Tree Frog
    "555086",  # Rusty Tree Frog
    # Aves
    "blkvul",  # Black Vulture
    "eardov1",  # Eared Dove
    "paldov1",  # Pale-vented Pigeon
    "neocor",  # Neotropic Cormorant
    "cogher1",  # Cocoi Heron
    "savhaw1",  # Savanna Hawk
    "blhpar1",  # Blue-headed Parrot
    "scamac1",  # Scarlet Macaw
    "gsflea1",  # Great-crested Flycatcher
    "cattyr",  # Cattle Tyrant
    "socfly1",  # Social Flycatcher (Added)
    "horcre1",  # House Wren
    "tropar",  # Tropical Parula
    "butsal1",  # Buff-throated Saltator
    # Insecta
    "1462737",  # Docidocercus fasciatus
    "1192948",  # Oxyprora surinamensis
    # Mammalia
    "47067",  # Brown-throated Three-toed Sloth
    "75154",  # Capybara
]

# check all of these are in the dataset
perch.filter(pl.col("species").is_in(list1)).group_by("species").agg(
    pl.count()
).collect().to_pandas()

/tmp/ipykernel_1121563/2033446926.py:29: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  perch.filter(pl.col("species").is_in(list1)).group_by("species").agg(pl.count()).collect().to_pandas()


,species,count
0,savhaw1,230
1,tropar,2578
2,555086,259
3,blhpar1,1006
4,eardov1,500
5,socfly1,3347
6,butsal1,2282
7,65344,92
8,47067,14
9,blkvul,402


In [5]:
list2 = [
    "65344",  # Boettger's Colombian Tree Frog (Amphibia)
    "555086",  # Rusty Tree Frog (Amphibia)
    "blkvul",  # Black Vulture (Aves)
    "eardov1",  # Eared Dove (Aves)
    "neocor",  # Neotropic Cormorant (Aves)
    "savhaw1",  # Savanna Hawk (Aves)
    "blhpar1",  # Blue-headed Parrot (Aves)
    "cattyr",  # Cattle Tyrant (Aves)
    "socfly1",  # Social Flycatcher (Aves)
    "tropar",  # Tropical Parula (Aves)
    "butsal1",  # Buff-throated Saltator (Aves)
    "1462737",  # Docidocercus fasciatus (Insecta)
    "1192948",  # Oxyprora surinamensis (Insecta)
    "47067",  # Brown-throated Three-toed Sloth (Mammalia)
    "strfly1",  # Streaked Flycatcher (Aves)
    "strcuc1",  # Striped Cuckoo (Aves)
    "41778",  # Neotropical River Otter (Mammalia)
    "42113",  # Collared Peccary (Mammalia)
    "67252",  # Veined Tree Frog (Amphibia)
    "52884",  # True Crickets (Insecta)
]

perch.filter(pl.col("species").is_in(list2)).group_by("species").agg(
    pl.count()
).collect().to_pandas()

/tmp/ipykernel_1121563/3430570373.py:25: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count()


,species,count
0,tropar,2578
1,52884,1191
2,socfly1,3347
3,1462737,140
4,blkvul,402
5,eardov1,500
6,42113,4
7,41778,40
8,neocor,491
9,butsal1,2282
